# Optimal Dispatch in einem 3-Knoten-Netz (lossless DC)

<a href="https://colab.research.google.com/github/USER/REPO/blob/main/simple_network_dispatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dieses Notebook löst das Vorlesungsbeispiel mit **Pyomo** und dem Solver **Gurobi**. Es läuft direkt in Google Colab — die Zelle unten installiert alles Nötige; Colab bringt bereits eine eingeschränkte Gurobi-Lizenz mit, die für dieses kleine Problem locker reicht.

## Problemstellung (Folie 1)

$$\min_{q} \; \sum_{i=1}^{N_k} mc_i(q_i) \cdot q_i$$

mit

$$\begin{aligned}
\sum_{i=1}^{N_k} q_i &= \sum_{i=1}^{N_k} d_i & \text{Energiebilanz (1)} \\
|H \cdot (q - d)| &\le t & \text{Leitungsgrenzen (2)} \\
q_i &\le c_i & \text{Erzeugungsgrenzen (3)}
\end{aligned}$$

Aus der Folie sind die konkreten Werte gegeben — die Grenzkostenfunktionen sind linear in $q$:

$$mc_1(q_1) = \tfrac{2\,\text{€}}{\text{MWh}^2} \cdot q_1, \qquad mc_2(q_2) = \tfrac{3{,}35\,\text{€}}{\text{MWh}^2} \cdot q_2, \qquad mc_3(q_3) = \tfrac{\infty\,\text{€}}{\text{MWh}}$$

Knoten 3 ist die Last ($c_3 = -25$ MWh, kein Erzeuger).

> **Hinweis zur Zielfunktion.** Bei linearen Grenzkosten $mc_i(q_i) = a_i q_i$ ist die **Erzeugungskostenfunktion** das Integral der Grenzkosten,
> $$C_i(q_i) = \int_0^{q_i} mc_i(s)\,ds = \tfrac{1}{2} a_i q_i^2.$$
> Wir minimieren $\sum_i C_i(q_i)$. Genau diese Konvention reproduziert die Folien-LMPs $p_i = mc_i(q_i^*)$, weil dann $\partial C_i / \partial q_i = mc_i(q_i)$ gilt — der LMP eines unbeschränkten Erzeugers ist also exakt seine momentane Grenzkostenrate. Das **quadratische Programm** ist somit
> $$\min \; \tfrac{1}{2} a_1\, q_1^2 + \tfrac{1}{2} a_2\, q_2^2 \quad \text{mit} \quad a_1 = 2, \; a_2 = 3{,}35.$$

## 1 — Setup

In [ ]:
# In Google Colab läuft diese Zelle in ~10 Sekunden.
# Pyomo und gurobipy werden installiert; Gurobi bringt seine eigene
# eingeschränkte Lizenz mit (genug für bis zu ~2000 Variablen).
%pip install -q pyomo gurobipy

import numpy as np
import pyomo.environ as pyo

## 2 — Daten

Wir tragen die Netzdaten genau so ein, wie sie in der Folie stehen. Die PTDF-Matrix $H$ ist bereits gegeben (Knoten 3 als Slack eliminiert), die Spalten entsprechen den Erzeugerknoten 1 und 2, die Zeilen den Leitungen $(1{-}2)$, $(1{-}3)$, $(2{-}3)$.

In [ ]:
GEN   = [1, 2]               # Knoten mit Erzeugung
NODES = [1, 2, 3]
LINES = [(1, 2), (1, 3), (2, 3)]

# Steigung der Grenzkostengeraden  mc_i(q) = a_i * q   [€/MWh^2]
a = {1: 2.00, 2: 3.35}

# Erzeugungskapazitäten c_i  [MWh]
c = {1: 30.0, 2: 30.0}

# Lasten d_i an jedem Knoten  [MWh]   (Knoten 3 hat c_3 = -25)
d = {1: 0.0, 2: 0.0, 3: 25.0}

# PTDF-Matrix H (Slack-Knoten 3 eliminiert)
# H_{l,i} = Anteil der Einspeisung am Knoten i, der über Leitung l fließt
H = {
    (1, 2): {1:  3/5, 2: -1/10},
    (1, 3): {1:  3/5, 2:  9/10},
    (2, 3): {1:  2/5, 2:  1/10},
}

# Thermische Leitungsgrenzen t_l  [MW]
t_lim = {(1, 2): 8, (1, 3): 30, (2, 3): 20}

# Erwartete Lösung (aus der Folie) zum Quervergleich
y_star_expected = {1: 15.0, 2: 10.0}
flows_expected  = {(1,2): 8.0, (1,3): 18.0, (2,3): 7.0}
lmp_expected    = {1: 30.0, 2: 33.5, 3: 33.0}

## 3 — Modell in Pyomo

Wir übersetzen die drei Restriktionen 1:1. Das `Suffix`-Objekt importiert die **Dualvariablen** $\lambda_0$ (zur Energiebilanz) und $\lambda_t$ (zu den Leitungsgrenzen) aus dem Solver — die brauchen wir später für die LMPs.

In [ ]:
m = pyo.ConcreteModel()

# Entscheidungsvariablen q_i mit 0 <= q_i <= c_i   (das ist Bedingung (3))
m.q = pyo.Var(GEN, domain=pyo.NonNegativeReals, bounds=lambda m, i: (0, c[i]))

# Zielfunktion:  sum_i C_i(q_i) = sum_i (1/2) * a_i * q_i^2   (Integral der Grenzkosten)
m.cost = pyo.Objective(expr=sum(0.5 * a[i] * m.q[i]**2 for i in GEN), sense=pyo.minimize)

# (1) Energiebilanz
m.balance = pyo.Constraint(expr=sum(m.q[i] for i in GEN) == sum(d.values()))

# (2) Leitungsgrenzen  -t <= H*q <= t   (d=0 an Erzeugerknoten, Slack=3 eliminiert)
def line_upper(m, i, j):
    return sum(H[(i, j)][k] * m.q[k] for k in GEN) <=  t_lim[(i, j)]
def line_lower(m, i, j):
    return sum(H[(i, j)][k] * m.q[k] for k in GEN) >= -t_lim[(i, j)]

m.line_upper = pyo.Constraint(LINES, rule=line_upper)
m.line_lower = pyo.Constraint(LINES, rule=line_lower)

# Dualvariablen einsammeln (für LMP-Berechnung)
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

## 4 — Lösen mit Gurobi

Gurobi löst dieses konvexe QP exakt und schnell. Wichtig: Die Option `QCPDual=1` weist Gurobi an, **Dualvariablen auch bei (quadratischen) konvexen Problemen** mitzuliefern — die brauchen wir gleich für die LMP-Berechnung.

In [ ]:
# 'gurobi_direct' spricht direkt mit dem gurobipy-Python-Modul --
# das funktioniert auf Colab ohne Binary im PATH.
solver = pyo.SolverFactory('gurobi_direct')
solver.options['QCPDual'] = 1   # Dualvariablen auch für konvexe QPs zurückgeben
result = solver.solve(m, tee=False)
print("Status:", result.solver.termination_condition)
print("Zielfunktionswert:", round(pyo.value(m.cost), 4), "[€/h]")

## 5 — Ergebnis: Einspeisung und Flüsse

In [ ]:
q_opt = {i: pyo.value(m.q[i]) for i in GEN}

print("Einspeisung q*:")
for i in GEN:
    print(f"  q_{i} = {q_opt[i]:6.3f} MWh   (Folie: {y_star_expected[i]})")

print("\nLeitungsflüsse  q_l = (H @ q)_l :")
for (i, j) in LINES:
    flow = sum(H[(i, j)][k] * q_opt[k] for k in GEN)
    active = "  <-- aktiv" if abs(abs(flow) - t_lim[(i, j)]) < 1e-4 else ""
    print(f"  q_{i}{j} = {flow:6.3f} MW   (Folie: {flows_expected[(i,j)]:5.2f},  Limit {t_lim[(i,j)]}){active}")

Das Limit auf Leitung $(1,2)$ ist **aktiv** ($q_{12} = 8$). Das ist der entscheidende Treiber des Beispiels: ohne diese Beschränkung würde der billige Erzeuger 1 (mc-Steigung 2) den teuren Erzeuger 2 (mc-Steigung 3,35) weiter verdrängen.

## 6 — Lokationale Strompreise (LMPs)

Laut Folie sind die LMPs die Dualvariablen des Optimierungsproblems:

$$LMP_{\text{slack}} = \lambda_0, \qquad LMP_i = \lambda_0 + \sum_{t=1}^{N_t} \lambda_t \cdot H_{(t,i)}$$

Knoten 3 ist der Slack-Knoten, also $LMP_3 = \lambda_0$. Für die Nicht-Slack-Knoten muss der Schattenpreis des betreffenden Leitungslimits über die PTDF-Spalte verrechnet werden. Vorzeichen: wir verwenden die Konvention $\lambda_t = \mu_{\text{upper}} - \mu_{\text{lower}}$, sodass ein bindendes oberes Limit positiv beiträgt.

In [ ]:
lambda0 = m.dual[m.balance]                       # zur Energiebilanz

# Schattenpreise der Leitungslimits (positives Vorzeichen für 'upper')
lambda_t = {}
for l in LINES:
    lambda_t[l] = m.dual[m.line_upper[l]] - m.dual[m.line_lower[l]]

print(f"λ_0 (Energiebilanz)   = {lambda0:7.4f}  €/MWh")
for l in LINES:
    print(f"λ_t auf Leitung {l} = {lambda_t[l]:7.4f}  €/MWh")

# LMPs: LMP_i = λ_0 + Σ_t λ_t * H_{t,i}
# Für Slack-Knoten (hier 3): LMP_3 = λ_0
lmp = {3: lambda0}
for i in GEN:
    lmp[i] = lambda0 + sum(lambda_t[l] * H[l][i] for l in LINES)

print("\nLokationale Strompreise:")
for i in NODES:
    print(f"  p_{i} = {lmp[i]:7.4f} €/MWh   (Folie: {lmp_expected[i]})")

**Sanity-Check** über die Grenzkosten der eingesetzten Erzeuger:

$$p_1 = mc_1(q_1^*) = 2 \cdot 15 = 30 \;\text{€/MWh}, \qquad p_2 = mc_2(q_2^*) = 3{,}35 \cdot 10 = 33{,}5 \;\text{€/MWh}.$$

Beide Erzeuger speisen *innerhalb* ihrer Kapazitäten ein, also muss ihr Preis ihrer Grenzkostenrate entsprechen. Der Preis am Lastknoten 3 liegt **zwischen** diesen beiden — das spiegelt wider, dass eine zusätzliche MWh Last an Knoten 3 anteilig durch beide Erzeuger gedeckt werden müsste, wobei die Engpassleitung $(1,2)$ die Aufteilung bestimmt.

## 7 — Was wir gelernt haben

* Das **Dispatch-Problem** lässt sich direkt aus der Folien-Notation als Pyomo-Modell formulieren — die drei Restriktionen $(1)$–$(3)$ werden 1:1 übersetzt.
* Weil $mc_i$ linear in $q_i$ ist, wird die Zielfunktion **quadratisch**. Gurobi löst dieses konvexe QP problemlos; ein reiner LP-Solver wie GLPK genügt nicht.
* Die **LMPs entstehen automatisch** als Dualvariablen — wir müssen sie nicht separat ausrechnen, der Solver liefert sie über das `Suffix`-Objekt mit.
* Ohne Netzengpass wäre $p_1 = p_2 = p_3$. **Erst die aktive Leitungsgrenze** $t_{12} \le 8$ macht die Preise lokational unterschiedlich — genau das ist der Punkt des Beispiels.

### Übungsvorschlag

Setze $t_{12} = 30$ (Limit faktisch deaktivieren) und löse erneut. Wie ändern sich $q^*$ und die LMPs? Wann genau wird der Unterschied zwischen den Knotenpreisen null?